# Split cgrp-gamma

In this notebook, we further subcluster the CGRP gamma neurons 

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [ ]:
from pathlib import Path
import sys
import os
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.patches as mpatches
import matplotlib.colors as clr
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
import math
from matplotlib.colors import Normalize
import matplotlib.cm as cm
import seaborn as sns

In [ ]:
plt.rcParams['figure.figsize'] = (3,3)
plt.rcParams["figure.dpi"] = 150

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))

from config.paths import BASE_DIR

input_dir = BASE_DIR / "data/h5ad/export_04/04b_reclustered"
output_dir = BASE_DIR / "data/h5ad/export_04/04c_gamma_subtypes"

output_dir.mkdir(parents=True, exist_ok=True)

input_file = input_dir / "neurons-seurat.h5ad"

In [ ]:
sc.settings.figdir = output_dir

In [ ]:
def assign_cell_type_colors(adata, key="cell_type"):
    """
    Assigns a tab10 color palette to a categorical obs field.
    If there are more than 10 categories, the palette cycles.
    """
    # Ensure categorical
    adata.obs[key] = adata.obs[key].astype("category")
    
    cats = adata.obs[key].cat.categories
    n = len(cats)

    # tab10 gives 10 colors; using modulo lets us cycle if n > 1=20
    base_palette = sns.color_palette("tab20", 20)

    palette = [mcolors.to_hex(base_palette[i % 20]) for i in range(n)]

    # assign colors in category order
    adata.uns[f"{key}_colors"] = palette

## Prepare data

In [ ]:
adata = sc.read_h5ad(input_file)
adata.obs.EGFP_dTomato_dual_hi.value_counts()

In [ ]:
sc.pl.umap(adata, color = 'seurat_labels', frameon = False)

In [ ]:
adata.X = adata.layers['log1p'].copy()

# Resolve G1/G2

In [ ]:
# 1. Subset to the CGRP-Gamma population
gamma_adata = adata[adata.obs['seurat_labels'] == 'CGRP-Gamma'].copy()

# 2. Use scVI latent space instead of PCA for neighbors
sc.pp.neighbors(gamma_adata, use_rep='X_scVI_seurat_neuron')
sc.tl.umap(gamma_adata)
sc.tl.leiden(gamma_adata, resolution=0.1, key_added='gamma_subclusters')

# 3. Identify G1 vs G2 based on Chrna3 mean expression
# Temporarily pull gene expression into .obs to allow grouping
gamma_adata.obs['Chrna3_temp'] = gamma_adata[:, 'Chrna3'].X.toarray().flatten()

cluster_expr = gamma_adata.obs.groupby('gamma_subclusters', observed=False)['Chrna3_temp'].mean()
g2_cluster = cluster_expr.idxmax()
g1_cluster = cluster_expr.idxmin()

# Clean up the temporary column to avoid plotting conflicts
del gamma_adata.obs['Chrna3_temp']

# 4. Create the final labels
label_map = {g1_cluster: 'CGRP-Gamma 1', g2_cluster: 'CGRP-Gamma 2'}
gamma_adata.obs['neuron_labels'] = gamma_adata.obs['gamma_subclusters'].map(label_map)

# Update the main adata object with the new sub-labels
adata.obs['neuron_labels'] = adata.obs['seurat_labels'].astype(str)
adata.obs.update(gamma_adata.obs[['neuron_labels']])
adata.obs['neuron_labels'] = adata.obs['neuron_labels'].astype('category')

# 5. Visualize to confirm the split
sc.pl.umap(gamma_adata, color=['gamma_subclusters', 'Chrna3', 'Adra2a', 'reporter_status'])

In [ ]:
sc.pl.matrixplot(gamma_adata, ['Nos1', 'Gpx3', 'Gfra3', 'Adra2a', 'Kcnk3', 
                                   'Hpse', 'Chrna3', 'Mgat5b', 'Col25a1', 'Mrap2'], groupby='neuron_labels', cmap = 'Blues')

In [ ]:
sc.pl.umap(gamma_adata, color=['Chrna3', 'Htr3a', 'Adra2a', 'Nos1'], frameon = False, ncols = 2)

In [ ]:
sc.pl.umap(gamma_adata, color=['gamma_subclusters', 'Chrna3', 'Adra2a', 'seurat_labels'], frameon = False)

# Map subclusters onto adata

In [ ]:
# 1. Start with the original labels in a new column
adata.obs['neuron_label'] = adata.obs['seurat_labels'].astype(str)

# 2. Map the subset subclusters to your G1/G2 nomenclature
# Based on your UMAP: Cluster 1 = Chrna3 high (G2), Cluster 0 = Adra2a high (G1)
gamma_map = {'1': 'CGRP-Gamma 2', '0': 'CGRP-Gamma 1'}
gamma_adata.obs['refined_label'] = gamma_adata.obs['gamma_subclusters'].map(gamma_map)

# 3. Update the main object's neuron_label column with the refined Gamma labels
# This only overwrites the cells present in gamma_adata
adata.obs.set_index(adata.obs.index, inplace=True) # Ensure indices match
adata.obs.update(gamma_adata.obs[['refined_label']].rename(columns={'refined_label': 'neuron_label'}))

# 4. Finalize as a category for better plotting
adata.obs['neuron_label'] = adata.obs['neuron_label'].astype('category')

In [ ]:
assign_cell_type_colors(adata,  key="neuron_label")

In [ ]:
sc.pl.umap(adata, color = 'neuron_label', frameon = False)

# Export

In [ ]:
adata_path = os.path.join(output_dir, 'neurons-final-labels.h5ad')
adata.write_h5ad(adata_path, compression='gzip')
print(adata_path)